# Lecture 11: 迁移学习与微调 (Transfer Learning and Fine-tuning)本笔记深入探讨迁移学习的理论和实践：预训练模型、域适应、渐进式解冻、特征提取 vs 微调。**学习目标：**- 理解为什么迁移学习如此有效- 区分特征提取和微调两种策略- 实现渐进式解冻训练- 掌握不同层的迁移效果分析- 理解域适应的基本原理**四步教学路径：** 直觉理解 -> 手动计算 -> 代码实现 -> 实验观察

## 目录1. 迁移学习概述与动机2. 特征提取 vs 微调：手动分析3. 预训练模型模拟与特征提取4. 微调实现与渐进式解冻5. 不同层迁移效果实验6. 学习率策略对比实验7. 域适应原理与实现8. 作业与参考文献

## 1. 迁移学习概述与动机### 什么是迁移学习？迁移学习将在大数据集（如ImageNet）上预训练的模型知识迁移到小数据集任务上。### 为什么有效？1. **低级特征通用**：边缘、纹理、颜色等特征在几乎所有视觉任务中都有用2. **高级特征可迁移**：物体部件、形状模式等也有一定通用性3. **数据效率**：小数据集无法从零训练深度网络，但迁移学习可以### 两种主要策略| 策略 | 冻结层 | 训练层 | 学习率 | 适用场景 ||---|---|---|---|---|| 特征提取 | 全部骨干 | 仅分类头 | 高(0.01) | 数据极少 || 微调 | 前几层 | 后几层+分类头 | 低(0.001) | 数据中等 || 全微调 | 无 | 全部 | 极低(0.0001) | 数据较多 || 渐进式解冻 | 分阶段 | 逐层解冻 | 逐渐降低 | 最佳实践 |### ImageNet预训练的威力- ImageNet有120万张图片，1000类- 预训练模型学到的特征层次：边缘 -> 纹理 -> 部件 -> 对象 -> 场景- 即使目标任务完全不同（如医学影像），低级特征仍然有用

## 2. 特征提取 vs 微调：手动分析### 特征提取冻结预训练模型的所有层，只用其输出特征训练一个新的分类器。**手动计算特征提取效果：**假设预训练模型输出512维特征：- 新任务有100个样本，5个类别- 训练 logistic regression: W(512x5) + b(5)- 参数量：512*5 + 5 = 2565- 对比从零训练：如果网络有100万参数，特征提取只需训练0.26%的参数### 微调解冻部分或全部预训练层，使用较小的学习率重新训练。**手动计算微调效果：**- 学习率策略：骨干层 lr=0.001，新分类头 lr=0.01- 梯度回传到骨干层，但更新幅度很小- 优点：适应新任务的特征分布- 风险：学习率太大会"灾难性遗忘"### 灾难性遗忘当学习率过大时，微调会破坏预训练学到的通用特征：- 原始特征：通用的边缘/纹理检测器- 过度微调后：特征退化，只适应当前小数据集- 表现：训练集精度高，但泛化能力下降**预防方法：**1. 使用小学习率（1-2个数量级低于从头训练）2. 使用差分学习率（深层小，浅层更小）3. 渐进式解冻4. 早停

In [5]:
# -*- coding: utf-8 -*-import numpy as npimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltnp.random.seed(42)# ============================================================# Simulate a pretrained model (randomly initialized but structured)# ============================================================class PretrainedModel:    # Simulates a pretrained CNN with multiple layers    def __init__(self, input_dim=64, hidden_dims=[128, 64, 32], n_classes=1000):        self.layers = []        prev = input_dim        for h in hidden_dims:            W = np.random.randn(prev, h) * np.sqrt(2.0 / prev)            b = np.zeros(h)            self.layers.append({'W': W, 'b': b, 'dim': h})            prev = h        # Original classification head (1000 classes)        self.W_head = np.random.randn(prev, n_classes) * 0.01        self.b_head = np.zeros(n_classes)        self.hidden_dims = hidden_dims        def extract_features(self, x, layer_idx=None):        # Extract features up to specified layer (or all)        a = x        features = [a]        for i, layer in enumerate(self.layers):            z = a @ layer['W'] + layer['b']            a = np.maximum(0, z)  # ReLU            features.append(a)            if layer_idx is not None and i == layer_idx:                return a        return a  # Last layer features        def forward(self, x):        feat = self.extract_features(x)        return feat @ self.W_head + self.b_head# Create pretrained modelpretrained = PretrainedModel(input_dim=64, hidden_dims=[128, 64, 32], n_classes=1000)print("Pretrained model created.")print(f"Layers: {len(pretrained.layers)}")for i, layer in enumerate(pretrained.layers):    print(f"  Layer {i}: {layer['W'].shape}")print(f"Classification head: {pretrained.W_head.shape}")

In [6]:
# ============================================================# Generate source and target domain data# ============================================================np.random.seed(42)# Source domain: 1000 samples, 1000 classes (simulating ImageNet)N_source = 1000X_source = np.random.randn(N_source, 64)# Make source data structuredW_source = np.random.randn(64, 1000) * 0.1y_source = (X_source @ W_source).argmax(1)# Target domain: 100 samples, 5 classes (simulating small custom dataset)N_target = 100C_target = 5# Create related but different distributionX_target = np.random.randn(N_target, 64) * 1.2 + 0.3  # Slightly shiftedW_target = np.random.randn(64, C_target) * 0.2y_target = (X_target @ W_target + np.random.randn(N_target, C_target) * 0.5).argmax(1)print(f"Source domain: {N_source} samples, 1000 classes, dim=64")print(f"Target domain: {N_target} samples, {C_target} classes, dim=64")print(f"Target class distribution: {np.bincount(y_target)}")

## 3. 特征提取：代码实现将预训练模型作为固定特征提取器，只训练新的分类头。

In [8]:
# ============================================================# Strategy 1: Feature Extraction (freeze all backbone layers)# ============================================================# Extract features using pretrained modeltrain_features = pretrained.extract_features(X_target)test_features = pretrained.extract_features(X_target[:20])  # Use first 20 as testprint(f"Extracted features shape: {train_features.shape}")# Train simple linear classifier on extracted featuresclass LinearClassifier:    def __init__(self, input_dim, output_dim, lr=0.01):        self.W = np.random.randn(input_dim, output_dim) * 0.01        self.b = np.zeros(output_dim)        self.lr = lr        def forward(self, x):        return x @ self.W + self.b        def train(self, X, y, n_epochs=200):        N = X.shape[0]        C = self.W.shape[1]        losses = []        accs = []        for epoch in range(n_epochs):            scores = self.forward(X)            shifted = scores - scores.max(1, keepdims=True)            probs = np.exp(shifted) / np.exp(shifted).sum(1, keepdims=True)            loss = -np.sum(np.log(probs[np.arange(N), y] + 1e-12)) / N            losses.append(loss)            accs.append((scores.argmax(1) == y).mean())                        dscores = probs.copy()            dscores[np.arange(N), y] -= 1            dscores /= N            dW = X.T @ dscores            db = dscores.sum(0)            self.W -= self.lr * dW            self.b -= self.lr * db        return losses, accs# Train feature extraction classifierclf_fe = LinearClassifier(32, C_target, lr=0.1)losses_fe, accs_fe = clf_fe.train(train_features, y_target, n_epochs=300)print(f"Feature Extraction - Final loss: {losses_fe[-1]:.4f}, Final acc: {accs_fe[-1]:.4f}")

## 4. 微调实现与渐进式解冻### 微调策略1. **全微调**：解冻所有层，使用小学习率2. **部分微调**：只解冻后几层，冻结前几层3. **渐进式解冻**：逐步解冻，每次解冻一层后训练几个epoch### 渐进式解冻算法```阶段1: 冻结骨干，只训练分类头 (epoch 1-10, lr=0.01)阶段2: 解冻最后一层骨干 (epoch 11-20, lr=0.005)  阶段3: 解冻倒数第二层 (epoch 21-30, lr=0.001)阶段4: 解冻所有层 (epoch 31+, lr=0.0001)```### 差分学习率不同层使用不同学习率：- 分类头：lr_head（最大）- 后几层骨干：lr_upper（中等）- 前几层骨干：lr_lower（最小）理由：浅层特征更通用，不需要大幅更新；深层特征需要适应新任务。

In [10]:
# ============================================================# Strategy 2: Fine-tuning with differential learning rates# ============================================================class FineTunedModel:    def __init__(self, pretrained_model, n_classes, strategy='full'):        # Copy pretrained weights        self.layers = []        for layer in pretrained_model.layers:            self.layers.append({                'W': layer['W'].copy(),                'b': layer['b'].copy()            })        # New classification head        last_dim = pretrained_model.layers[-1]['W'].shape[1]        self.W_head = np.random.randn(last_dim, n_classes) * 0.01        self.b_head = np.zeros(n_classes)                # Learning rates based on strategy        if strategy == 'full':            self.lrs = [0.0001] * len(self.layers) + [0.01]  # backbone + head        elif strategy == 'partial':            self.lrs = [0.0] * (len(self.layers) - 1) + [0.0001, 0.01]        elif strategy == 'differential':            self.lrs = []            for i in range(len(self.layers)):                self.lrs.append(0.0001 * (0.5 ** (len(self.layers) - 1 - i)))            self.lrs.append(0.01)        def forward(self, x):        a = x        self.acts = [a]        self.pre_acts = []        for layer in self.layers:            z = a @ layer['W'] + layer['b']            self.pre_acts.append(z)            a = np.maximum(0, z)            self.acts.append(a)        self.scores = a @ self.W_head + self.b_head        return self.scores        def backward(self, x, y):        N = x.shape[0]        C = self.W_head.shape[1]        shifted = self.scores - self.scores.max(1, keepdims=True)        probs = np.exp(shifted) / np.exp(shifted).sum(1, keepdims=True)        loss = -np.sum(np.log(probs[np.arange(N), y] + 1e-12)) / N                dscores = probs.copy()        dscores[np.arange(N), y] -= 1        dscores /= N                dW_head = self.acts[-1].T @ dscores        db_head = dscores.sum(0)                da = dscores @ self.W_head.T        grads = []        for i in range(len(self.layers) - 1, -1, -1):            drelu = da * (self.pre_acts[i] > 0).astype(float)            dW = self.acts[i].T @ drelu            db = drelu.sum(0)            grads.append((dW, db))            da = drelu @ self.layers[i]['W'].T        grads.reverse()                # Update with differential learning rates        all_params = [(self.layers[i]['W'], self.layers[i]['b'], grads[i][0], grads[i][1]) for i in range(len(self.layers))]        all_params.append((self.W_head, self.b_head, dW_head, db_head))                for i, (W, b, dW, db) in enumerate(all_params):            W -= self.lrs[i] * dW            b -= self.lrs[i] * db                return loss        def train(self, X, y, n_epochs=300):        losses = []        accs = []        for epoch in range(n_epochs):            self.forward(X)            loss = self.backward(X, y)            losses.append(loss)            pred = self.scores.argmax(1)            accs.append((pred == y).mean())        return losses, accs# Train with different strategiesmodel_full = FineTunedModel(pretrained, C_target, strategy='full')model_partial = FineTunedModel(pretrained, C_target, strategy='partial')model_diff = FineTunedModel(pretrained, C_target, strategy='differential')losses_full, accs_full = model_full.train(X_target, y_target, n_epochs=300)losses_partial, accs_partial = model_partial.train(X_target, y_target, n_epochs=300)losses_diff, accs_diff = model_diff.train(X_target, y_target, n_epochs=300)print(f"Full fine-tune: loss={losses_full[-1]:.4f}, acc={accs_full[-1]:.4f}")print(f"Partial fine-tune: loss={losses_partial[-1]:.4f}, acc={accs_partial[-1]:.4f}")print(f"Differential LR: loss={losses_diff[-1]:.4f}, acc={accs_diff[-1]:.4f}")

In [11]:
# ============================================================# Compare strategies# ============================================================fig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].plot(losses_fe, label='Feature Extraction', linewidth=2, alpha=0.8)axes[0].plot(losses_full, label='Full Fine-tune', linewidth=2, alpha=0.8)axes[0].plot(losses_partial, label='Partial Fine-tune', linewidth=2, alpha=0.8)axes[0].plot(losses_diff, label='Differential LR', linewidth=2, alpha=0.8)axes[0].set_xlabel('Epoch')axes[0].set_ylabel('Loss')axes[0].set_title('Training Loss', fontweight='bold')axes[0].legend(fontsize=9)axes[0].grid(True, alpha=0.3)axes[1].plot(accs_fe, label='Feature Extraction', linewidth=2, alpha=0.8)axes[1].plot(accs_full, label='Full Fine-tune', linewidth=2, alpha=0.8)axes[1].plot(accs_partial, label='Partial Fine-tune', linewidth=2, alpha=0.8)axes[1].plot(accs_diff, label='Differential LR', linewidth=2, alpha=0.8)axes[1].set_xlabel('Epoch')axes[1].set_ylabel('Accuracy')axes[1].set_title('Training Accuracy', fontweight='bold')axes[1].legend(fontsize=9)axes[1].grid(True, alpha=0.3)plt.suptitle('Transfer Learning Strategies Comparison', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-11-transfer-learning/strategies.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 1 saved: strategies.png")

## 5. 不同层迁移效果实验### 实验设计从预训练模型的不同层提取特征，观察哪一层的特征对目标任务最有效。### 预期结果- 过浅的特征（如第1层）：太底层，缺乏语义信息- 过深的特征（如最后一层）：可能过于特化于源任务- 中间层特征：通常效果最好（"最佳迁移层"）

In [13]:
# ============================================================# Experiment: Transfer from different layers# ============================================================layer_accs = []for layer_idx in range(len(pretrained.layers)):    # Extract features from this layer    feats = pretrained.extract_features(X_target, layer_idx=layer_idx)        # Train classifier    clf = LinearClassifier(feats.shape[1], C_target, lr=0.1)    _, accs = clf.train(feats, y_target, n_epochs=200)    layer_accs.append(accs[-1])    print(f"Layer {layer_idx} ({feats.shape[1]} dims): final acc = {accs[-1]:.4f}")# Also try raw input featuresclf_raw = LinearClassifier(64, C_target, lr=0.1)_, accs_raw = clf_raw.train(X_target, y_target, n_epochs=200)layer_accs.insert(0, accs_raw[-1])fig, ax = plt.subplots(figsize=(10, 5))layer_names = ['Raw input'] + [f'Layer {i}' for i in range(len(pretrained.layers))]ax.bar(layer_names, layer_accs, color='steelblue', edgecolor='black', alpha=0.8)ax.set_xlabel('Feature Layer')ax.set_ylabel('Final Accuracy')ax.set_title('Transfer from Different Layers', fontsize=14, fontweight='bold')ax.grid(True, alpha=0.3, axis='y')for i, acc in enumerate(layer_accs):    ax.text(i, acc + 0.01, f'{acc:.3f}', ha='center', fontsize=10)plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-11-transfer-learning/layer_transfer.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 2 saved: layer_transfer.png")

## 6. 渐进式解冻实现### 算法详解渐进式解冻（Gradual Unfreezing）是ULMFiT提出的技术：1. **阶段1（epoch 0-5）**：冻结所有骨干层，只训练分类头2. **阶段2（epoch 6-10）**：解冻最后一层骨干，连同分类头一起训练3. **阶段3（epoch 11-15）**：解冻倒数第二层4. **...**：继续逐层解冻### 为什么有效？1. **防止灾难性遗忘**：先让分类头适应特征，再缓慢调整特征2. **稳定性**：避免一开始就大幅修改预训练权重3. **类似课程学习**：从简单（固定特征）到复杂（全部微调）### 手动计算假设3层网络，每层学习率：- 阶段1：lr = [0, 0, 0, 0.01]（只有分类头更新）- 阶段2：lr = [0, 0, 0.001, 0.01]（解冻layer2）- 阶段3：lr = [0, 0.0005, 0.001, 0.01]（解冻layer1）- 阶段4：lr = [0.0001, 0.0005, 0.001, 0.01]（全部解冻）学习率随层加深而减小，这叫**分层学习率衰减**。

In [15]:
# ============================================================# Progressive Unfreezing Implementation# ============================================================class ProgressiveUnfreezing:    def __init__(self, pretrained_model, n_classes, n_layers):        self.layers = []        for layer in pretrained_model.layers:            self.layers.append({'W': layer['W'].copy(), 'b': layer['b'].copy()})        last_dim = pretrained_model.layers[-1]['W'].shape[1]        self.W_head = np.random.randn(last_dim, n_classes) * 0.01        self.b_head = np.zeros(n_classes)        self.n_layers = n_layers        self.unfrozen_layers = 0  # Start with 0 backbone layers unfrozen        def forward(self, x):        self.a = [x]        self.z_list = []        a = x        for layer in self.layers:            z = a @ layer['W'] + layer['b']            self.z_list.append(z)            a = np.maximum(0, z)            self.a.append(a)        self.scores = a @ self.W_head + self.b_head        return self.scores        def train_epoch(self, X, y):        N = X.shape[0]        self.forward(X)        shifted = self.scores - self.scores.max(1, keepdims=True)        probs = np.exp(shifted) / np.exp(shifted).sum(1, keepdims=True)        loss = -np.sum(np.log(probs[np.arange(N), y] + 1e-12)) / N        acc = (self.scores.argmax(1) == y).mean()                dscores = probs.copy()        dscores[np.arange(N), y] -= 1        dscores /= N                dW_head = self.a[-1].T @ dscores        db_head = dscores.sum(0)                # Always update head        self.W_head -= 0.01 * dW_head        self.b_head -= 0.01 * db_head                # Backprop through unfrozen layers        da = dscores @ self.W_head.T        for i in range(self.n_layers - 1, max(self.n_layers - 1 - self.unfrozen_layers, -1), -1):            drelu = da * (self.z_list[i] > 0).astype(float)            lr = 0.001 * (0.5 ** (self.n_layers - 1 - i))            dW = self.a[i].T @ drelu            db = drelu.sum(0)            self.layers[i]['W'] -= lr * dW            self.layers[i]['b'] -= lr * db            da = drelu @ self.layers[i]['W'].T                return loss, acc        def unfreeze_one_more(self):        if self.unfrozen_layers < self.n_layers:            self.unfrozen_layers += 1            print(f"  Unfroze layer {self.n_layers - self.unfrozen_layers} (total unfrozen: {self.unfrozen_layers})")# Train with progressive unfreezingmodel_pu = ProgressiveUnfreezing(pretrained, C_target, len(pretrained.layers))losses_pu = []accs_pu = []unfreeze_epochs = [0, 50, 100, 150]  # Unfreeze at these epochsfor epoch in range(250):    if epoch in unfreeze_epochs:        model_pu.unfreeze_one_more()    loss, acc = model_pu.train_epoch(X_target, y_target)    losses_pu.append(loss)    accs_pu.append(acc)print(f"Progressive Unfreezing - Final loss: {losses_pu[-1]:.4f}, Final acc: {accs_pu[-1]:.4f}")

In [16]:
# ============================================================# Visualize progressive unfreezing# ============================================================fig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].plot(losses_pu, linewidth=2, color='purple')for ue in unfreeze_epochs[1:]:    axes[0].axvline(x=ue, color='red', linestyle='--', alpha=0.5, label='Unfreeze' if ue == unfreeze_epochs[1] else '')axes[0].set_xlabel('Epoch')axes[0].set_ylabel('Loss')axes[0].set_title('Progressive Unfreezing Loss', fontweight='bold')axes[0].grid(True, alpha=0.3)if unfreeze_epochs[1:]:    axes[0].legend()axes[1].plot(accs_pu, linewidth=2, color='green')for ue in unfreeze_epochs[1:]:    axes[1].axvline(x=ue, color='red', linestyle='--', alpha=0.5, label='Unfreeze' if ue == unfreeze_epochs[1] else '')axes[1].set_xlabel('Epoch')axes[1].set_ylabel('Accuracy')axes[1].set_title('Progressive Unfreezing Accuracy', fontweight='bold')axes[1].grid(True, alpha=0.3)if unfreeze_epochs[1:]:    axes[1].legend()plt.suptitle('Progressive Unfreezing Training Curve', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-11-transfer-learning/progressive.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 3 saved: progressive.png")

## 7. 域适应 (Domain Adaptation)### 直觉域适应处理源域和目标域分布不同的情况。例如：源域是合成图片，目标域是真实图片。### 核心问题源域 D_S 有标签，目标域 D_T 无标签（或极少标签），且 P_S != P_T。### 方法：域对抗训练 (DANN)1. 特征提取器 F：将输入映射到特征空间2. 标签分类器 C：预测类别标签3. 域判别器 D：区分源域和目标域**关键技巧：梯度反转层 (GRL)**- 前向传播：GRL是恒等映射- 反向传播：GRL将梯度取反效果：特征提取器学习"欺骗"域判别器，使特征域不变。### 手动计算假设源域特征均值=[1, 1]，目标域特征均值=[3, 3]：- 域适应前：域判别器可以轻松区分- 域适应后：特征提取器调整，使两域特征均值都接近[2, 2]- 结果：域判别器无法区分，标签分类器在两域上都有效

In [18]:
# ============================================================# Domain Adaptation: Feature alignment visualization# ============================================================np.random.seed(42)# Source domain: Gaussian centered at [1, 1]source = np.random.randn(200, 2) * 0.5 + np.array([1, 1])# Target domain: Gaussian centered at [3, 3]target = np.random.randn(200, 2) * 0.5 + np.array([3, 3])# Simulate domain adaptation: gradually align distributionsfig, axes = plt.subplots(1, 3, figsize=(15, 4))# Before adaptationaxes[0].scatter(source[:, 0], source[:, 1], c='blue', alpha=0.5, label='Source', s=20)axes[0].scatter(target[:, 0], target[:, 1], c='red', alpha=0.5, label='Target', s=20)axes[0].set_title('Before Adaptation', fontweight='bold')axes[0].legend()axes[0].grid(True, alpha=0.3)# During adaptation (partial alignment)alpha = 0.5  # Adaptation strengthsource_partial = source * (1 - alpha) + np.array([2, 2]) * alphatarget_partial = target * (1 - alpha) + np.array([2, 2]) * alphaaxes[1].scatter(source_partial[:, 0], source_partial[:, 1], c='blue', alpha=0.5, label='Source', s=20)axes[1].scatter(target_partial[:, 0], target_partial[:, 1], c='red', alpha=0.5, label='Target', s=20)axes[1].set_title('During Adaptation (50%)', fontweight='bold')axes[1].legend()axes[1].grid(True, alpha=0.3)# After adaptation (full alignment)mean_center = (source.mean(0) + target.mean(0)) / 2source_aligned = source - source.mean(0) + mean_centertarget_aligned = target - target.mean(0) + mean_centeraxes[2].scatter(source_aligned[:, 0], source_aligned[:, 1], c='blue', alpha=0.5, label='Source', s=20)axes[2].scatter(target_aligned[:, 0], target_aligned[:, 1], c='red', alpha=0.5, label='Target', s=20)axes[2].set_title('After Adaptation', fontweight='bold')axes[2].legend()axes[2].grid(True, alpha=0.3)plt.suptitle('Domain Adaptation: Feature Alignment', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-11-transfer-learning/domain_adapt.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 4 saved: domain_adapt.png")# Compute Maximum Mean Discrepancy (MMD) as domain distance metricdef mmd(X, Y, kernel='rbf', gamma=1.0):    # Simple MMD estimation    XX = np.mean(np.exp(-gamma * np.sum((X[:, None] - X[None, :])**2, axis=2)))    YY = np.mean(np.exp(-gamma * np.sum((Y[:, None] - Y[None, :])**2, axis=2)))    XY = np.mean(np.exp(-gamma * np.sum((X[:, None] - Y[None, :])**2, axis=2)))    return XX + YY - 2 * XYmmd_before = mmd(source, target)mmd_after = mmd(source_aligned, target_aligned)print(f"MMD before adaptation: {mmd_before:.4f}")print(f"MMD after adaptation:  {mmd_after:.4f}")print(f"MMD reduction: {mmd_before / max(mmd_after, 1e-10):.1f}x")

## 7.5 迁移学习数据量分析### 数据量与策略选择| 目标数据量 | 推荐策略 | 典型场景 ||---|---|---|| < 100样本 | 特征提取 | 医学影像（罕见疾病） || 100-1000样本 | 微调（差分LR） | 工业缺陷检测 || 1000-10000样本 | 全微调 | 细粒度分类 || > 10000样本 | 从头训练 | 大规模分类 |### 手动计算：数据量vs准确率假设：- 从头训练：acc = 1 - 1/sqrt(N)，需要大量数据- 特征提取：acc = 0.7 + 0.2*tanh(N/100)，小数据也较好- 微调：acc = 0.8 + 0.15*tanh(N/500)，中等数据最佳

In [20]:
# ============================================================# Data size vs strategy comparison# ============================================================data_sizes = [10, 20, 50, 100, 200, 500, 1000, 2000, 5000]accs_from_scratch = [1 - 1/np.sqrt(n) for n in data_sizes]accs_feat_ext = [0.7 + 0.2 * np.tanh(n/100) for n in data_sizes]accs_finetune = [0.8 + 0.15 * np.tanh(n/500) for n in data_sizes]fig, ax = plt.subplots(figsize=(10, 6))ax.plot(data_sizes, accs_from_scratch, 'o-', linewidth=2, label='From scratch')ax.plot(data_sizes, accs_feat_ext, 's-', linewidth=2, label='Feature extraction')ax.plot(data_sizes, accs_finetune, '^-', linewidth=2, label='Fine-tuning')ax.set_xlabel('Number of training samples')ax.set_ylabel('Accuracy')ax.set_title('Data Size vs Strategy', fontsize=14, fontweight='bold')ax.legend(fontsize=11)ax.set_xscale('log')ax.grid(True, alpha=0.3)ax.axvspan(10, 100, alpha=0.1, color='red', label='Feature extraction zone')ax.axvspan(100, 1000, alpha=0.1, color='yellow', label='Fine-tuning zone')ax.axvspan(1000, 5000, alpha=0.1, color='green', label='Full training zone')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-11-transfer-learning/data_size.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 5 saved: data_size.png")

## 8. 学习率对微调的影响### 直觉微调时学习率是关键超参数：- 学习率过大 -> 破坏预训练特征（灾难性遗忘）- 学习率过小 -> 收敛太慢，无法适应新任务### 经验法则- 微调学习率通常是从头训练的1/10到1/100- 分类头使用更高学习率（新初始化）- 使用warmup避免初期不稳定

In [22]:
# ============================================================# Learning rate effect on fine-tuning# ============================================================lrs = [0.00001, 0.0001, 0.001, 0.01, 0.1, 1.0]lr_results = {}for lr_val in lrs:    np.random.seed(42)    model = FineTunedModel(pretrained, C_target, strategy='full')    model.lrs = [lr_val * 0.01] * len(model.layers) + [lr_val]    losses_lr, accs_lr = model.train(X_target, y_target, n_epochs=200)    lr_results[lr_val] = {'loss': losses_lr, 'acc': accs_lr}fig, axes = plt.subplots(1, 2, figsize=(14, 5))for lr_val, res in lr_results.items():    axes[0].plot(res['loss'], linewidth=2, label=f'lr={lr_val}', alpha=0.8)    axes[1].plot(res['acc'], linewidth=2, label=f'lr={lr_val}', alpha=0.8)axes[0].set_xlabel('Epoch')axes[0].set_ylabel('Loss')axes[0].set_title('Fine-tuning Loss at Different LR', fontweight='bold')axes[0].legend(fontsize=9)axes[0].grid(True, alpha=0.3)axes[1].set_xlabel('Epoch')axes[1].set_ylabel('Accuracy')axes[1].set_title('Fine-tuning Accuracy at Different LR', fontweight='bold')axes[1].legend(fontsize=9)axes[1].grid(True, alpha=0.3)plt.suptitle('Learning Rate Effect on Fine-tuning', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-11-transfer-learning/lr_effect.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 6 saved: lr_effect.png")for lr_val, res in lr_results.items():    print(f"lr={lr_val}: final_loss={res['loss'][-1]:.4f}, final_acc={res['acc'][-1]:.4f}")

## 8.5 灾难性遗忘实验### 实验设计1. 在源任务上训练模型2. 在目标任务上微调（不同学习率）3. 回到源任务测试，观察精度下降程度### 预期- 学习率越大，源任务精度下降越多（遗忘越严重）- 渐进式解冻能缓解遗忘

In [24]:
# ============================================================# Catastrophic Forgetting Experiment# ============================================================np.random.seed(42)# Train on source task firstmodel_cf = FineTunedModel(pretrained, 1000, strategy='full')source_acc_before = []for epoch in range(50):    model_cf.forward(X_source[:200])    loss = model_cf.backward(X_source[:200], y_source[:200])    pred = model_cf.scores.argmax(1)    source_acc_before.append((pred == y_source[:200]).mean())source_acc_before_final = source_acc_before[-1]print(f"Source task accuracy before fine-tuning: {source_acc_before_final:.4f}")# Now fine-tune on target task with different LRslrs_test = [0.0001, 0.001, 0.01, 0.1, 1.0]forgetting_results = []for lr_val in lrs_test:    np.random.seed(42)    model_ft = FineTunedModel(pretrained, C_target, strategy='full')    model_ft.lrs = [lr_val * 0.01] * len(model_ft.layers) + [lr_val]    model_ft.train(X_target, y_target, n_epochs=200)        # Test on source task (using original head dimensions)    source_feats = model_ft.forward(X_source[:200])    # Re-create source classifier    W_src = np.random.randn(source_feats.shape[1], 1000) * 0.01    src_scores = source_feats @ W_src    src_pred = src_scores.argmax(1)    source_acc_after = (src_pred == y_source[:200]).mean()        forgetting = source_acc_before_final - source_acc_after    forgetting_results.append({'lr': lr_val, 'source_acc': source_acc_after, 'forgetting': forgetting})    print(f"lr={lr_val}: source_acc_after={source_acc_after:.4f}, forgetting={forgetting:.4f}")fig, ax = plt.subplots(figsize=(10, 5))lrs_plot = [r['lr'] for r in forgetting_results]forgetting_plot = [r['forgetting'] for r in forgetting_results]ax.bar(range(len(lrs_plot)), forgetting_plot, color='salmon', edgecolor='black', alpha=0.8)ax.set_xticks(range(len(lrs_plot)))ax.set_xticklabels([str(l) for l in lrs_plot])ax.set_xlabel('Fine-tuning Learning Rate')ax.set_ylabel('Forgetting (accuracy drop)')ax.set_title('Catastrophic Forthing vs Learning Rate', fontsize=14, fontweight='bold')ax.grid(True, alpha=0.3, axis='y')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-11-transfer-learning/forgetting.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 7 saved: forgetting.png")

## 8.6 特征冻结实验：层间相似度分析### 直觉预训练模型的不同层学到的特征有多通用？我们可以通过分析层间特征相似度来回答。### 方法1. 计算源域和目标域在第i层的特征2. 计算两个域特征的CKA（Centered Kernel Alignment）相似度3. 相似度高的层 -> 特征更通用 -> 适合冻结4. 相似度低的层 -> 特征更特化 -> 需要微调### 手动计算CKACKA(K, L) = ||K^T L||_F / (||K||_F * ||L||_F)其中 K 和 L 是特征矩阵的核矩阵。

In [26]:
# ============================================================# Layer similarity analysis (simplified CKA)# ============================================================def feature_similarity(X_source, X_target, model, layer_idx):    # Extract features from both domains    feat_s = model.extract_features(X_source, layer_idx=layer_idx)    feat_t = model.extract_features(X_target, layer_idx=layer_idx)        # Compute mean feature distance between domains    mean_diff = np.linalg.norm(feat_s.mean(0) - feat_t.mean(0))    # Normalize by source feature norm    norm = np.linalg.norm(feat_s.mean(0)) + np.linalg.norm(feat_t.mean(0))    similarity = 1 - mean_diff / (norm + 1e-8)    return similarity# Compute similarity for each layersimilarities = []for i in range(len(pretrained.layers)):    sim = feature_similarity(X_source[:100], X_target, pretrained, i)    similarities.append(sim)    print(f"Layer {i}: domain similarity = {sim:.4f}")fig, ax = plt.subplots(figsize=(8, 5))ax.plot(range(len(similarities)), similarities, 'o-', linewidth=2, markersize=8, color='purple')ax.set_xlabel('Layer Index')ax.set_ylabel('Source-Target Similarity')ax.set_title('Feature Similarity Across Layers', fontsize=14, fontweight='bold')ax.grid(True, alpha=0.3)ax.axhline(y=0.9, color='green', linestyle='--', alpha=0.5, label='High similarity (freeze)')ax.axhline(y=0.7, color='red', linestyle='--', alpha=0.5, label='Low similarity (fine-tune)')ax.legend()plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-11-transfer-learning/layer_sim.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 8 saved: layer_sim.png")

## 8.7 Mixup 数据增强在迁移学习中的应用### 直觉Mixup通过线性插值两个样本及其标签来生成新样本，提升泛化能力：x_mix = lambda * x_a + (1 - lambda) * x_by_mix = lambda * y_a + (1 - lambda) * y_b### 在迁移学习中的优势1. 平滑决策边界，减少过拟合2. 特别适合小数据集微调3. 与特征提取兼容

In [28]:
# ============================================================# Mixup augmentation experiment# ============================================================def mixup_data(x, y, alpha=0.2):    # Generate mixup samples    lam = np.random.beta(alpha, alpha)    batch_size = x.shape[0]    indices = np.random.permutation(batch_size)    mixed_x = lam * x + (1 - lam) * x[indices]    mixed_y = (lam * y, (1 - lam) * y[indices])    return mixed_x, mixed_y, lam# One-hot encode labelsy_onehot = np.zeros((N_target, C_target))y_onehot[np.arange(N_target), y_target] = 1# Train with and without mixupnp.random.seed(42)model_no_mixup = FineTunedModel(pretrained, C_target, strategy='differential')losses_no_mx, accs_no_mx = model_no_mixup.train(X_target, y_target, n_epochs=200)np.random.seed(42)model_mixup = FineTunedModel(pretrained, C_target, strategy='differential')losses_mx, accs_mx = [], []for epoch in range(200):    x_mix, (y_a, y_b), lam = mixup_data(X_target, y_onehot, alpha=0.2)    model_mixup.forward(x_mix)    # Mixed loss    N = x_mix.shape[0]    shifted = model_mixup.scores - model_mixup.scores.max(1, keepdims=True)    probs = np.exp(shifted) / np.exp(shifted).sum(1, keepdims=True)    loss = -lam * np.sum(np.log(probs[np.arange(N), y_a.argmax(1)] + 1e-12)) / N    loss -= (1 - lam) * np.sum(np.log(probs[np.arange(N), y_b.argmax(1)] + 1e-12)) / N    losses_mx.append(loss)    accs_mx.append((model_mixup.scores.argmax(1) == y_target).mean())        # Backward with mixed labels    dscores = probs.copy()    dscores[np.arange(N), y_a.argmax(1)] -= lam    dscores[np.arange(N), y_b.argmax(1)] -= (1 - lam)    dscores /= N        dW_head = model_mixup.acts[-1].T @ dscores    db_head = dscores.sum(0)        da = dscores @ model_mixup.W_head.T    for i in range(len(model_mixup.layers) - 1, -1, -1):        drelu = da * (model_mixup.pre_acts[i] > 0).astype(float)        lr = model_mixup.lrs[i]        model_mixup.layers[i]['W'] -= lr * (model_mixup.acts[i].T @ drelu)        model_mixup.layers[i]['b'] -= lr * drelu.sum(0)        da = drelu @ model_mixup.layers[i]['W'].T    model_mixup.W_head -= model_mixup.lrs[-1] * dW_head    model_mixup.b_head -= model_mixup.lrs[-1] * db_headfig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].plot(losses_no_mx, label='No Mixup', linewidth=2)axes[0].plot(losses_mx, label='With Mixup', linewidth=2)axes[0].set_xlabel('Epoch')axes[0].set_ylabel('Loss')axes[0].set_title('Training Loss', fontweight='bold')axes[0].legend()axes[0].grid(True, alpha=0.3)axes[1].plot(accs_no_mx, label='No Mixup', linewidth=2)axes[1].plot(accs_mx, label='With Mixup', linewidth=2)axes[1].set_xlabel('Epoch')axes[1].set_ylabel('Accuracy')axes[1].set_title('Training Accuracy', fontweight='bold')axes[1].legend()axes[1].grid(True, alpha=0.3)plt.suptitle('Mixup Augmentation in Transfer Learning', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-11-transfer-learning/mixup.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 9 saved: mixup.png")print(f"No Mixup: final acc={accs_no_mx[-1]:.4f}")print(f"Mixup:    final acc={accs_mx[-1]:.4f}")

## 8.8 综合实验：所有策略对比将所有迁移学习策略放在一张图中对比，方便决策。

In [30]:
# ============================================================# Comprehensive comparison of all strategies# ============================================================strategies = {    'Feature Extraction': (losses_fe, accs_fe),    'Full Fine-tune': (losses_full, accs_full),    'Partial Fine-tune': (losses_partial, accs_partial),    'Differential LR': (losses_diff, accs_diff),    'Progressive Unfreeze': (losses_pu, accs_pu),}fig, axes = plt.subplots(1, 2, figsize=(16, 6))for name, (losses_s, accs_s) in strategies.items():    # Pad shorter sequences    max_len = max(len(losses_s) for _, (losses_s, _) in strategies.items())    padded_loss = losses_s + [losses_s[-1]] * (max_len - len(losses_s))    padded_acc = accs_s + [accs_s[-1]] * (max_len - len(accs_s))    axes[0].plot(padded_loss, linewidth=2, label=name, alpha=0.8)    axes[1].plot(padded_acc, linewidth=2, label=name, alpha=0.8)axes[0].set_xlabel('Epoch')axes[0].set_ylabel('Loss')axes[0].set_title('All Strategies: Loss', fontweight='bold')axes[0].legend(fontsize=9)axes[0].grid(True, alpha=0.3)axes[1].set_xlabel('Epoch')axes[1].set_ylabel('Accuracy')axes[1].set_title('All Strategies: Accuracy', fontweight='bold')axes[1].legend(fontsize=9)axes[1].grid(True, alpha=0.3)plt.suptitle('Comprehensive Transfer Learning Comparison', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-11-transfer-learning/all_strategies.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 10 saved: all_strategies.png")print("\n=== Final Results ===")for name, (_, accs_s) in strategies.items():    print(f"{name}: {accs_s[-1]:.4f}")

## 8.9 迁移学习实践检查清单### 训练前检查1. **预训练模型选择**：   - 任务相似度高 -> 同领域预训练模型   - 任务相似度低 -> ImageNet通用预训练   2. **数据量评估**：   - < 100样本 -> 特征提取   - 100-1000 -> 微调   - > 1000 -> 全微调或从头训练3. **预处理一致性**：   - 确保使用与预训练模型相同的归一化   - 相同的输入尺寸（或允许的尺寸范围）### 训练中监控1. **训练损失下降**：如果损失不降，检查学习率2. **验证集精度**：监控过拟合3. **源任务精度**：检查灾难性遗忘4. **梯度统计**：各层梯度量级不应相差过大### 训练后分析1. **特征可视化**：用t-SNE查看目标域特征分布2. **错误分析**：分析哪些类别迁移效果好/差3. **层重要性**：分析哪些层贡献最大

In [32]:
# ============================================================# Final summary table visualization# ============================================================fig, ax = plt.subplots(figsize=(12, 6))ax.axis('off')strat_names = ['FeatureExtraction', 'PartialFine-tune', 'FullFine-tune', 'DifferentialLR', 'ProgressiveUnfreeze']final_accs = [accs_fe[-1], accs_partial[-1], accs_full[-1], accs_diff[-1], accs_pu[-1]]colors_bar = ['skyblue', 'lightgreen', 'orange', 'salmon', 'purple']bars = ax.bar(range(len(strat_names)), final_accs, color=colors_bar, edgecolor='black', alpha=0.8)ax.set_xticks(range(len(strat_names)))ax.set_xticklabels(strat_names, fontsize=10)ax.set_ylabel('Final Accuracy', fontsize=12)ax.set_title('Transfer Learning Strategies: Final Accuracy', fontsize=14, fontweight='bold')ax.grid(True, alpha=0.3, axis='y')for bar, acc in zip(bars, final_accs):    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,             f'{acc:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part3-frontiers/lecture-11-transfer-learning/final_summary.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 11 saved: final_summary.png")

## 9. 关键要点总结| 策略 | 何时使用 | 优点 | 风险 ||---|---|---|---|| 特征提取 | 数据<100 | 安全、快速 | 无法适应新分布 || 部分微调 | 数据100-1000 | 平衡适应与稳定 | 需调超参 || 全微调 | 数据>1000 | 最大适应能力 | 灾难性遗忘风险 || 渐进式解冻 | 通用 | 最稳定 | 训练时间较长 || 差分学习率 | 通用 | 精细控制 | 需调多个LR |**最佳实践：**1. 从特征提取开始作为baseline2. 数据够多时切换到微调3. 使用差分学习率（深层小，浅层更小）4. 考虑渐进式解冻5. 使用warmup6. 监控源任务精度防止遗忘

## 10. 作业### 作业1：实现SLR (Stochastic Layer Rehabilitation)SLR是渐进式解冻的变体，不是完全冻结/解冻，而是用概率控制：- 每层有一个"活跃概率" p_i- 前向传播时以概率p_i使用预训练权重- 训练过程中p_i从0逐渐增大到1**要求：**1. 实现SLR训练2. 与渐进式解冻对比3. 分析哪种方法收敛更稳定4. 绘制学习曲线对比图

### 作业2：实现MMD域适应使用Maximum Mean Discrepancy (MMD)作为域距离度量，通过最小化MMD实现域适应：**要求：**1. 实现基于MMD的域适应2. 使用RBF核和多项式核3. 对比不同gamma值的效果4. 分析MMD与DANN的优劣

### 作业3：Low-Shot Learning实验在极小数据集（每类1-5个样本）上测试迁移学习：**要求：**1. 每类1/3/5/10个样本对比2. 特征提取 vs 微调 vs Episode-based训练3. 分析few-shot vs many-shot的最优策略差异4. 实现Prototypical Networks（用类均值作为原型）

## 11. 参考文献1. [[Yosinski et al., 2014]](https://arxiv.org/abs/1411.1792) - How transferable are features in deep neural networks?2. [[Howard and Ruder, 2018]](https://arxiv.org/abs/1801.06146) - Universal Language Model Fine-tuning for Text Classification (ULMFiT, progressive unfreezing)3. [[Ganin et al., 2016]](https://arxiv.org/abs/1505.07818) - Domain-Adversarial Training of Neural Networks (DANN)4. [[Long et al., 2015]](https://arxiv.org/abs/1502.02791) - Learning Transferable Features with Deep Adaptation Networks (MMD)5. [[Kornblith et al., 2019]](https://arxiv.org/abs/1912.05777) - Do Better ImageNet Models Transfer Better?6. [[He et al., 2019]](https://arxiv.org/abs/1812.08743) - Bag of Tricks for Image Classification with Convolutional Neural Networks7. [[Snell et al., 2017]](https://arxiv.org/abs/1703.05175) - Prototypical Networks for Few-shot Learning8. [[French, 1999]](https://www.cs.cornell.edu/~carpenter/bliss/language_learning/french_catastrophic_1999.pdf) - Catastrophic forgetting in connectionist networks9. [[Oquab et al., 2014]](https://arxiv.org/abs/1405.1803) - Learning and Transferring Mid-level Image Representations10. [[CS231n, Stanford]](https://cs231n.github.io/transfer-learning/) - CS231n: Transfer Learning

---

## 参考代码实现

以下 GitHub 仓库提供了本节内容的完整代码实现，建议结合学习：

- **[huggingface/pytorch-image-models](https://github.com/huggingface/pytorch-image-models)** (37139 stars): 预训练模型库（迁移学习基础）
  - 仓库地址: https://github.com/huggingface/pytorch-image-models

- **[seloufian/Deep-Learning-Computer-Vision](https://github.com/seloufian/Deep-Learning-Computer-Vision)** (135 stars): 迁移学习与微调实践
  - 仓库地址: https://github.com/seloufian/Deep-Learning-Computer-Vision


> 标注说明: 以上仓库按热度排序，优先推荐 stars 最多的实现。

